# LeetCode 1160 — Find Words That Can Be Formed by Characters bằng Python

**Link đề:** https://leetcode.com/problems/find-words-that-can-be-formed-by-characters/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String, Counting

```python
class Solution:
    def countCharacters(self, words: List[str], chars: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu bài toán "**có đủ nguyên liệu không?**": mỗi ký tự trong `chars` chỉ dùng **một lần cho mỗi từ**.
2. Viết baseline "nhặt từng chữ" bằng list và `remove`, thấy chỗ lãng phí.
3. Học kỹ thuật **đếm tần suất** (frequency counting) bằng `dict`, bằng **mảng 26 phần tử** (`ord`) và bằng `collections.Counter`.
4. Điều kiện cốt lõi: với mọi chữ `c`, **số lần `c` trong từ ≤ số lần `c` trong chars**.
5. Trace, chứng minh, test đối chiếu, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và ví dụ

Cho mảng chuỗi `words` và chuỗi `chars`. Một chuỗi là **good** nếu có thể tạo nó từ các ký tự trong `chars` (mỗi ký tự trong `chars` chỉ dùng **một lần** cho **mỗi** từ). Trả về **tổng độ dài** của mọi chuỗi good.

| `words` | `chars` | Output | Giải thích |
|---|---|---:|---|
| `["cat","bt","hat","tree"]` | `"atach"` | `6` | "cat" (3) + "hat" (3) |
| `["hello","world","leetcode"]` | `"welldonehoneyr"` | `10` | "hello" + "world" |

Vì sao `"bt"` không good? `chars` không có chữ `b`. Vì sao `"tree"` không good? `chars` không có chữ `r`, cũng không có chữ `e` nào.

**Constraints:**
- `1 <= words.length <= 1000`
- `1 <= words[i].length, chars.length <= 100`
- Chỉ chữ cái tiếng Anh viết thường.

Lưu ý quan trọng: "dùng một lần **cho mỗi từ**" — mỗi từ được kiểm tra **độc lập**, `chars` được "làm mới" cho từ tiếp theo.

In [1]:
from typing import List

words = ["cat", "bt", "hat", "tree"]
chars = "atach"
print("chars có những chữ:", sorted(chars))
for w in words:
    print(w, "->", sorted(w))

chars có những chữ: ['a', 'a', 'c', 'h', 't']
cat -> ['a', 'c', 't']
bt -> ['b', 't']
hat -> ['a', 'h', 't']
tree -> ['e', 'e', 'r', 't']


## 2. Nền tảng Python cần biết

### 2.1. `list(s)` và `list.remove(x)`

- `list("atach")` → `['a','t','a','c','h']`.
- `lst.remove(x)` xóa **lần xuất hiện đầu tiên** của `x`; nếu không có `x` → `ValueError`.

In [2]:
pool = list("atach")
pool.remove("a")
print(pool)             # chỉ mất 1 chữ 'a'
try:
    pool.remove("b")
except ValueError as e:
    print("ValueError:", e)

['t', 'a', 'c', 'h']
ValueError: list.remove(x): x not in list


### 2.2. Đếm tần suất bằng `dict`

Ý tưởng: duyệt chuỗi, mỗi chữ `ch` thì `freq[ch] = freq.get(ch, 0) + 1`.
- `d.get(key, default)` trả `d[key]` nếu có, ngược lại trả `default` (không lỗi).

In [3]:
freq = {}
for ch in "atach":
    freq[ch] = freq.get(ch, 0) + 1
print(freq)                      # {'a': 2, 't': 1, 'c': 1, 'h': 1}
print(freq.get("a", 0), freq.get("z", 0))

{'a': 2, 't': 1, 'c': 1, 'h': 1}
2 0


### 2.3. Mảng 26 phần tử + `ord`

Chữ thường `'a'..'z'` ánh xạ sang index `0..25` bằng `ord(ch) - ord('a')`. `ord` trả mã số Unicode của ký tự.

In [4]:
print(ord("a"), ord("b"), ord("z"))
cnt = [0] * 26
for ch in "atach":
    cnt[ord(ch) - ord("a")] += 1
print(cnt)
print("số chữ 'a' =", cnt[0], "| số chữ 't' =", cnt[ord('t') - ord('a')])

97 98 122
[2, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
số chữ 'a' = 2 | số chữ 't' = 1


### 2.4. `collections.Counter`

`Counter(s)` là dict đặc biệt đếm sẵn tần suất. Truy cập key không tồn tại trả `0` (không lỗi). Từ Python 3.10, có thể so sánh `Counter(w) <= Counter(chars)` nghĩa là "mọi tần suất bên trái đều ≤ bên phải".

In [5]:
from collections import Counter

c_chars = Counter("atach")
print(c_chars)
print(c_chars["a"], c_chars["z"])        # 2 0
print(Counter("cat") - c_chars)          # phép trừ chỉ giữ số dương → Counter() rỗng nghĩa là đủ
print(Counter("bt") - c_chars)           # còn thiếu b

Counter({'a': 2, 't': 1, 'c': 1, 'h': 1})
2 0
Counter()
Counter({'b': 1})


`Counter(a) - Counter(b)` giữ lại các chữ mà `a` **nhiều hơn** `b` (số dương). Kết quả **rỗng** ⇔ `a` không cần chữ nào vượt quá `b` ⇔ từ good.

### 2.5. `all(...)` với generator

`all(điều_kiện for x in ...)` trả `True` nếu **mọi** phần tử thỏa; dừng ngay khi gặp phần tử sai.

In [6]:
print(all(x > 0 for x in [1, 2, 3]))     # True
print(all(x > 0 for x in [1, -2, 3]))    # False
print(all(x > 0 for x in []))            # True (không có phản ví dụ)

True
False
True


## 3. Cách 1 — Baseline: nhặt từng chữ khỏi "kho"

Với mỗi từ: copy `chars` thành list `pool`. Duyệt từng chữ của từ, nếu chữ có trong `pool` thì `remove`; không có thì từ không good.

In [7]:
def count_characters_remove(words: List[str], chars: str) -> int:
    total = 0
    for w in words:
        pool = list(chars)          # kho mới cho MỖI từ
        ok = True
        for ch in w:
            if ch in pool:
                pool.remove(ch)
            else:
                ok = False
                break
        if ok:
            total += len(w)
    return total

print(count_characters_remove(["cat", "bt", "hat", "tree"], "atach"))                      # 6
print(count_characters_remove(["hello", "world", "leetcode"], "welldonehoneyr"))           # 10

6
10


### Nút thắt
- `ch in pool` và `pool.remove(ch)` đều **duyệt list** → `O(C)` mỗi lần, với `C = len(chars)`.
- Mỗi từ dài `L` → `O(L · C)`; tạo `pool` cũng `O(C)`.
- Tổng: `O(N · L · C)` ≈ `1000 · 100 · 100 = 10^7` — vẫn qua được, nhưng ta **đếm lại `chars` cho từng từ** dù `chars` không đổi.

## 4. Insight — So sánh tần suất

Từ `w` good ⇔ **với mọi chữ `c`: `count_w[c] <= count_chars[c]`**.

- Đếm `chars` **một lần duy nhất**.
- Với mỗi từ, đếm tần suất của nó rồi so từng chữ.

In [8]:
c_chars = Counter("atach")
for w in ["cat", "bt", "hat", "tree"]:
    cw = Counter(w)
    detail = {c: (cw[c], c_chars[c]) for c in cw}
    good = all(cw[c] <= c_chars[c] for c in cw)
    print(f"{w:<5} cần/có = {detail} -> good = {good}")

cat   cần/có = {'c': (1, 1), 'a': (1, 2), 't': (1, 1)} -> good = True
bt    cần/có = {'b': (1, 0), 't': (1, 1)} -> good = False
hat   cần/có = {'h': (1, 1), 'a': (1, 2), 't': (1, 1)} -> good = True
tree  cần/có = {'t': (1, 1), 'r': (1, 0), 'e': (2, 0)} -> good = False


## 5. Cách 2 — Lời giải chính: mảng đếm 26 phần tử

**Mổ xẻ:**
1. `base = [0]*26`, đếm `chars` một lần.
2. Với mỗi từ `w`: `need = [0]*26`, đếm `w`.
3. Duyệt 26 chữ: nếu có chữ `need[i] > base[i]` → không good.
4. Good → cộng `len(w)`.

Có thể **dừng sớm** ngay khi đếm: vừa tăng `need[i]` mà vượt `base[i]` thì loại luôn.

In [9]:
def count_characters_array(words: List[str], chars: str) -> int:
    base = [0] * 26
    for ch in chars:
        base[ord(ch) - 97] += 1          # 97 = ord('a')

    total = 0
    for w in words:
        need = [0] * 26
        good = True
        for ch in w:
            i = ord(ch) - 97
            need[i] += 1
            if need[i] > base[i]:        # vượt quá số chữ đang có → loại sớm
                good = False
                break
        if good:
            total += len(w)
    return total

print(count_characters_array(["cat", "bt", "hat", "tree"], "atach"))
print(count_characters_array(["hello", "world", "leetcode"], "welldonehoneyr"))

6
10


## 6. TRACE từng bước

`chars = "atach"` → `base`: a=2, c=1, h=1, t=1.

| từ | chữ đang xét | need sau khi tăng | base | vượt? | kết luận |
|---|---|---|---|---|---|
| cat | c | c=1 | c=1 | không | |
| | a | a=1 | a=2 | không | |
| | t | t=1 | t=1 | không | good, +3 |
| bt | b | b=1 | b=0 | **có** | loại |
| hat | h, a, t | 1,1,1 | 1,2,1 | không | good, +3 |
| tree | t | t=1 | 1 | không | |
| | r | r=1 | 0 | **có** | loại |

Tổng = 6.

In [10]:
def trace_count_characters(words: List[str], chars: str) -> int:
    base = Counter(chars)
    print("base =", dict(base))
    total = 0
    for w in words:
        need = Counter()
        good = True
        for ch in w:
            need[ch] += 1
            if need[ch] > base[ch]:
                print(f"  {w!r}: chữ {ch!r} cần {need[ch]} > có {base[ch]} -> LOẠI")
                good = False
                break
        if good:
            total += len(w)
            print(f"  {w!r}: đủ chữ -> GOOD, cộng {len(w)}, total = {total}")
    return total

trace_count_characters(["cat", "bt", "hat", "tree"], "atach")
print()
trace_count_characters(["hello", "world", "leetcode"], "welldonehoneyr")

base = {'a': 2, 't': 1, 'c': 1, 'h': 1}
  'cat': đủ chữ -> GOOD, cộng 3, total = 3
  'bt': chữ 'b' cần 1 > có 0 -> LOẠI
  'hat': đủ chữ -> GOOD, cộng 3, total = 6
  'tree': chữ 'r' cần 1 > có 0 -> LOẠI

base = {'w': 1, 'e': 3, 'l': 2, 'd': 1, 'o': 2, 'n': 2, 'h': 1, 'y': 1, 'r': 1}
  'hello': đủ chữ -> GOOD, cộng 5, total = 5
  'world': đủ chữ -> GOOD, cộng 5, total = 10
  'leetcode': chữ 't' cần 1 > có 0 -> LOẠI


10

Chú ý `"hello"`: chữ `l` cần **2** lần — `chars = "welldonehoneyr"` có đúng 2 chữ `l` nên vẫn good. Đây là lý do phải so **tần suất**, không chỉ "có mặt hay không".

## 7. Cách 3 — `Counter` gọn gàng

- `Counter(w) - base` rỗng ⇔ good (chạy mọi phiên bản Python 3).
- Hoặc `Counter(w) <= base` (Python 3.10+).

In [11]:
def count_characters_counter(words: List[str], chars: str) -> int:
    base = Counter(chars)
    return sum(len(w) for w in words if not (Counter(w) - base))

print(count_characters_counter(["cat", "bt", "hat", "tree"], "atach"))
print(count_characters_counter(["hello", "world", "leetcode"], "welldonehoneyr"))

6
10


`not (Counter(...))`: một Counter/dict **rỗng** được coi là `False` trong ngữ cảnh điều kiện, nên `not rỗng` là `True`.

## 8. Cách 4 — `str.count` cho từng chữ khác nhau

Với mỗi chữ khác nhau `c` trong `set(w)`: so `w.count(c) <= chars.count(c)`. Code rất ngắn, mỗi `count` là `O(độ dài)`, tối đa 26 chữ khác nhau → chấp nhận được.

In [12]:
def count_characters_strcount(words: List[str], chars: str) -> int:
    total = 0
    for w in words:
        if all(w.count(c) <= chars.count(c) for c in set(w)):
            total += len(w)
    return total

print(count_characters_strcount(["cat", "bt", "hat", "tree"], "atach"))
print(set("hello"))     # set: tập các phần tử KHÔNG trùng

6
{'l', 'h', 'o', 'e'}


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def countCharacters(self, words: List[str], chars: str) -> int:
        base = [0] * 26
        for ch in chars:
            base[ord(ch) - 97] += 1
        total = 0
        for w in words:
            need = [0] * 26
            for ch in w:
                i = ord(ch) - 97
                need[i] += 1
                if need[i] > base[i]:
                    break
            else:                       # chỉ chạy khi vòng for KHÔNG bị break
                total += len(w)
        return total

sol = Solution()
print(sol.countCharacters(["cat", "bt", "hat", "tree"], "atach"))
print(sol.countCharacters(["hello", "world", "leetcode"], "welldonehoneyr"))

6
10


### `for ... else` — cú pháp hay bị hiểu nhầm

Khối `else` gắn với `for` chạy **khi vòng lặp kết thúc tự nhiên** (không gặp `break`). Ở đây: không break = không có chữ nào thiếu = từ good. Thay thế cho biến cờ `good = True/False`.

In [14]:
for x in [1, 2, 3]:
    if x == 5:
        break
else:
    print("không gặp break → else chạy")

for x in [1, 5, 3]:
    if x == 5:
        print("gặp 5, break")
        break
else:
    print("dòng này KHÔNG in")

không gặp break → else chạy
gặp 5, break


In [15]:
class SolutionCounter:
    def countCharacters(self, words: List[str], chars: str) -> int:
        base = Counter(chars)
        return sum(len(w) for w in words if not (Counter(w) - base))

print(SolutionCounter().countCharacters(["cat", "bt", "hat", "tree"], "atach"))

6


## 10. Vì sao đúng?

**Mệnh đề:** `w` tạo được từ `chars` ⇔ với mọi chữ `c`, `count_w(c) <= count_chars(c)`.
- (⇒) Nếu tạo được, mỗi lần dùng chữ `c` cho `w` tiêu tốn một bản `c` khác nhau trong `chars`, nên cần ít nhất `count_w(c)` bản.
- (⇐) Nếu đủ cho mọi chữ, ta gán lần lượt các bản `c` trong `chars` cho các vị trí `c` trong `w` — thứ tự không quan trọng vì đề chỉ yêu cầu "dùng ký tự", không yêu cầu giữ thứ tự.

**Invariant trong vòng lặp từ:** sau khi xử lý `k` chữ đầu của `w`, `need[i]` = số chữ `i` trong `w[:k]`, và mọi `need[i] <= base[i]` (nếu không đã break). Kết thúc không break ⇒ điều kiện đúng cho toàn bộ `w`.

Mỗi từ dùng `need` **mới** nên các từ độc lập đúng như đề.

## 11. Độ phức tạp

Gọi `N = len(words)`, `L` = tổng độ dài mọi từ, `C = len(chars)`.

| Cách | Time | Space |
|---|---|---|
| Nhặt từng chữ (`remove`) | `O(N·C + L·C)` | `O(C)` |
| Mảng 26 | `O(C + L + 26N)` = `O(C + L)` (vì `26N <= 26L`) | `O(26)` = `O(1)` |
| `Counter` | `O(C + L)` trung bình | `O(26)` |
| `str.count` | `O(L·26 + 26·C·N)` xấu nhất | `O(26)` |

Lời giải mảng 26 là **tuyến tính theo tổng input**.

## 12. Bộ test: mẫu, biên, phản ví dụ, đối chiếu

In [16]:
cases = [
    (["cat", "bt", "hat", "tree"], "atach", 6),
    (["hello", "world", "leetcode"], "welldonehoneyr", 10),
    (["a"], "a", 1),
    (["a"], "b", 0),
    (["aa"], "a", 0),                  # cần 2 chữ a, chỉ có 1
    (["aa", "a"], "aa", 3),            # mỗi từ độc lập, chars "làm mới"
    (["abc", "cba", "bca"], "abc", 9),  # thứ tự không quan trọng
    (["xyz"], "zyxw", 3),
    (["z" * 100], "z" * 99, 0),
]
solvers = [count_characters_remove, count_characters_array, count_characters_counter,
           count_characters_strcount, Solution().countCharacters, SolutionCounter().countCharacters]
for words, chars, expected in cases:
    for f in solvers:
        assert f(words, chars) == expected, (f, words, chars)
    print(f"{str(words)[:30]:<32} chars={chars[:15]!r:<18} -> {expected} PASS")

['cat', 'bt', 'hat', 'tree']     chars='atach'            -> 6 PASS
['hello', 'world', 'leetcode']   chars='welldonehoneyr'   -> 10 PASS
['a']                            chars='a'                -> 1 PASS
['a']                            chars='b'                -> 0 PASS
['aa']                           chars='a'                -> 0 PASS
['aa', 'a']                      chars='aa'               -> 3 PASS
['abc', 'cba', 'bca']            chars='abc'              -> 9 PASS
['xyz']                          chars='zyxw'             -> 3 PASS
['zzzzzzzzzzzzzzzzzzzzzzzzzzzz   chars='zzzzzzzzzzzzzzz'  -> 0 PASS


In [17]:
import random
random.seed(1160)
alphabet = "abcde"
for _ in range(500):
    chars = "".join(random.choice(alphabet) for _ in range(random.randint(1, 8)))
    words = ["".join(random.choice(alphabet) for _ in range(random.randint(1, 5))) for _ in range(random.randint(1, 6))]
    expected = count_characters_remove(words, chars)
    for f in solvers[1:]:
        assert f(words, chars) == expected
print("500 test ngẫu nhiên khớp baseline. PASS")

500 test ngẫu nhiên khớp baseline. PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra "có mặt", không kiểm tra số lượng
```python
all(ch in chars for ch in w)      # "aa" với chars "a" → True (SAI)
```

### Lỗi 2: Dùng chung một "kho" cho mọi từ
Xóa chữ khỏi `pool` rồi để nguyên cho từ sau → các từ sau bị thiếu chữ oan.

### Lỗi 3: Dùng `set(chars)` — mất thông tin số lần.

### Lỗi 4: Cộng **số từ** thay vì **tổng độ dài**.

In [18]:
def wrong_presence(words, chars):
    return sum(len(w) for w in words if all(ch in chars for ch in w))

def wrong_shared_pool(words, chars):
    pool = list(chars)                 # tạo MỘT lần
    total = 0
    for w in words:
        ok = True
        for ch in w:
            if ch in pool:
                pool.remove(ch)
            else:
                ok = False
                break
        if ok:
            total += len(w)
    return total

def wrong_count_words(words, chars):
    base = Counter(chars)
    return sum(1 for w in words if not (Counter(w) - base))

print("đúng:", Solution().countCharacters(["aa"], "a"), "| chỉ kiểm có mặt:", wrong_presence(["aa"], "a"))
print("đúng:", Solution().countCharacters(["aa", "a"], "aa"), "| kho dùng chung:", wrong_shared_pool(["aa", "a"], "aa"))
print("đúng:", Solution().countCharacters(["cat", "hat"], "atach"), "| đếm số từ:", wrong_count_words(["cat", "hat"], "atach"))

đúng: 0 | chỉ kiểm có mặt: 2
đúng: 3 | kho dùng chung: 2
đúng: 6 | đếm số từ: 2


## 14. Mẹo nhận dạng pattern

- "Có tạo được A từ các chữ của B không?", "đủ nguyên liệu không?", "anagram", "hoán vị" → **so sánh bảng tần suất**.
- Bảng chữ cái nhỏ cố định (26 chữ thường) → **mảng 26** nhanh và gọn; bảng chữ lớn/không biết trước → `dict`/`Counter`.
- Có dữ liệu **dùng lại nhiều lần** (`chars`) → đếm **một lần** ở ngoài vòng lặp.

Bài cùng họ: 383 (Ransom Note), 242 (Valid Anagram), 1189 (Maximum Number of Balloons), 2287.

## 15. Biến thể

### 15.1. Ransom Note (383) — chỉ một từ, trả True/False

In [19]:
def can_construct(ransom_note: str, magazine: str) -> bool:
    return not (Counter(ransom_note) - Counter(magazine))

print(can_construct("aa", "aab"), can_construct("aa", "ab"))

True False


### 15.2. Dùng chung kho cho mọi từ (khác đề!)
Nếu **mỗi ký tự chỉ được dùng một lần cho tất cả các từ**, bài thành "chọn tập con từ" — khó hơn nhiều (tối ưu cần duyệt tổ hợp/quy hoạch). Đây là lý do chữ "for each word" trong đề rất quan trọng.

### 15.3. Tạo được bao nhiêu bản của một từ? (giống 1189 Maximum Number of Balloons)
Số bản = `min(count_chars[c] // count_w[c])` trên các chữ `c` của từ.

In [20]:
def max_copies(word: str, chars: str) -> int:
    have, need = Counter(chars), Counter(word)
    return min(have[c] // need[c] for c in need)

print(max_copies("balloon", "nlaebolko"))         # 1
print(max_copies("balloon", "loonbalxballpoon"))  # 2

1
2


### 15.4. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| `remove` | nhặt chữ khỏi list | `O(L·C)` | `O(C)` | dễ hình dung |
| Mảng 26 | đếm + so sánh | `O(C + L)` | `O(1)` | **nộp bài** |
| `Counter` | phép trừ multiset | `O(C + L)` | `O(1)` | gọn nhất |
| `str.count` | đếm từng chữ khác nhau | `O(26·(L + N·C))` | `O(1)` | ngắn, hơi chậm |

## 16. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
`words = ["dog", "good", "god", "do"]`, `chars = "godo"`. Từ nào good? Tổng bằng bao nhiêu?

### Bài 2 — Đếm tần suất bằng dict (không dùng Counter)

In [21]:
def freq_exercise(s: str) -> dict:
    # TODO: dùng d.get(ch, 0) + 1
    pass

# assert freq_exercise("hello") == {"h": 1, "e": 1, "l": 2, "o": 1}

### Bài 3 — Hoàn thiện lời giải mảng 26

In [22]:
def count_characters_exercise(words: List[str], chars: str) -> int:
    # TODO 1: base = [0] * 26, đếm chars
    # TODO 2: với mỗi w, đếm need và dừng sớm khi vượt base
    # TODO 3: cộng len(w) nếu good
    pass

# assert count_characters_exercise(["cat", "bt", "hat", "tree"], "atach") == 6
# assert count_characters_exercise(["aa", "a"], "aa") == 3

### Bài 4 — Ransom Note bằng mảng 26

In [23]:
def ransom_note_exercise(note: str, magazine: str) -> bool:
    # TODO
    pass

# assert ransom_note_exercise("a", "b") == False
# assert ransom_note_exercise("aa", "aab") == True

### Bài 5 — Trả về danh sách các từ good (giữ thứ tự)

In [24]:
def good_words_exercise(words: List[str], chars: str) -> List[str]:
    # TODO
    pass

# assert good_words_exercise(["cat", "bt", "hat", "tree"], "atach") == ["cat", "hat"]

### Bài 6 — Mini challenge: chữ còn thiếu
Với một từ không good, trả về dict `{chữ: số_lượng_còn_thiếu}`.

In [25]:
def missing_letters_exercise(word: str, chars: str) -> dict:
    # TODO: với mỗi chữ c của word: thiếu = need[c] - have[c] nếu > 0
    pass

# assert missing_letters_exercise("tree", "atach") == {"r": 1, "e": 2}
# assert missing_letters_exercise("cat", "atach") == {}

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đếm chars: g1 o2 d1 | "good" cần o2 d1 g1 | so need ≤ have cho từng từ rồi cộng độ dài |
| 2 | dict rỗng | `for ch in s` | `d[ch] = d.get(ch, 0) + 1` |
| 3 | xem mục 5 | `ord(ch) - 97` | `if need[i] > base[i]: break` + `for/else` |
| 4 | chỉ 1 từ | đếm magazine | trừ dần, âm thì False |
| 5 | giống bài chính | list kết quả | `append(w)` thay vì cộng độ dài |
| 6 | Counter trừ Counter | `Counter(word) - Counter(chars)` | `dict(...)` |

## 18. Đáp án tham khảo

**Bài 1:** `chars = "godo"` → g=1, o=2, d=1. "dog" ✔ (3), "good" ✔ cần g1 o2 d1 (4), "god" ✔ (3), "do" ✔ (2) → tổng **12**.

In [26]:
def freq_answer(s: str) -> dict:
    d = {}
    for ch in s:
        d[ch] = d.get(ch, 0) + 1
    return d

def count_characters_answer(words: List[str], chars: str) -> int:
    base = [0] * 26
    for ch in chars:
        base[ord(ch) - 97] += 1
    total = 0
    for w in words:
        need = [0] * 26
        for ch in w:
            i = ord(ch) - 97
            need[i] += 1
            if need[i] > base[i]:
                break
        else:
            total += len(w)
    return total

def ransom_note_answer(note: str, magazine: str) -> bool:
    cnt = [0] * 26
    for ch in magazine:
        cnt[ord(ch) - 97] += 1
    for ch in note:
        i = ord(ch) - 97
        cnt[i] -= 1
        if cnt[i] < 0:
            return False
    return True

def good_words_answer(words: List[str], chars: str) -> List[str]:
    base = Counter(chars)
    return [w for w in words if not (Counter(w) - base)]

def missing_letters_answer(word: str, chars: str) -> dict:
    return dict(Counter(word) - Counter(chars))

assert Solution().countCharacters(["dog", "good", "god", "do"], "godo") == 12
assert freq_answer("hello") == {"h": 1, "e": 1, "l": 2, "o": 1}
assert count_characters_answer(["cat", "bt", "hat", "tree"], "atach") == 6
assert count_characters_answer(["aa", "a"], "aa") == 3
assert ransom_note_answer("a", "b") == False
assert ransom_note_answer("aa", "aab") == True
assert good_words_answer(["cat", "bt", "hat", "tree"], "atach") == ["cat", "hat"]
assert missing_letters_answer("tree", "atach") == {"r": 1, "e": 2}
assert missing_letters_answer("cat", "atach") == {}
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countCharacters(self, words: List[str], chars: str) -> int:
        base = [0] * 26
        for ch in chars:
            base[ord(ch) - 97] += 1
        total = 0
        for w in words:
            need = [0] * 26
            for ch in w:
                i = ord(ch) - 97
                need[i] += 1
                if need[i] > base[i]:
                    break
            else:
                total += len(w)
        return total
```

**Template so tần suất:** `not (Counter(need) - Counter(have))`.

**Checklist:**
- [ ] So **số lượng** từng chữ, không chỉ "có mặt"?
- [ ] Mỗi từ có bộ đếm `need` **mới** (kho `chars` không bị tiêu hao giữa các từ)?
- [ ] Cộng **độ dài** từ good, không cộng số từ?
- [ ] Test từ có chữ lặp (`"aa"`), từ chứa chữ không có trong chars?